<a href="https://colab.research.google.com/github/micah-shull/AI_Agents/blob/main/902_MSOv3_UnitTests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Your testing file is **very solid**. You’re testing exactly the right layer: the **utilities**, not the graph. That’s the correct approach for an agent architecture like yours.

You’re essentially testing the **three pillars of the system**:

```
Telemetry → Metrics → Reporting
```

Which means if these tests pass, your orchestrator is almost guaranteed to behave correctly.

I'll break down:

1. What you're doing right
2. One small structural improvement
3. Two high-value tests I recommend adding

---

# 1. What You're Doing Exactly Right

### Testing utilities instead of the graph

You correctly wrote:

```
Use minimal or mock data; no graph
```

That’s perfect.

Testing the graph itself is usually unnecessary because:

```
graph = plumbing
utilities = logic
```

Your logic lives in:

```
load_telemetry
compute_signature_metrics
build_report_sections
assemble_report
```

So that’s where tests belong.

---

# 2. Your Test Coverage Is Well Structured

You naturally divided the tests into:

```
Telemetry
Metrics
Reporting
```

Which mirrors the architecture.

This is ideal because the pipeline becomes:

```
Raw Data
  ↓
Telemetry Tests
  ↓
Metrics Tests
  ↓
Reporting Tests
```

If anything breaks, you know **exactly where it broke**.

---

# 3. Telemetry Tests Are Good

This test is especially important:

```python
test_load_telemetry_loads_runs_and_filters_by_mission_id
```

You're verifying two critical behaviors:

1️⃣ JSON loading works
2️⃣ mission filtering works

Filtering bugs are extremely common in telemetry pipelines, so testing this is excellent.

---

### The empty data directory test is also good

```
test_load_telemetry_empty_data_dir
```

This ensures your system behaves safely when data is missing.

Which is very important for production systems.

Your test confirms:

```
mission_runs == []
run_count == 0
telemetry_summary exists
```

That’s exactly the behavior you want.

---

# 4. Metrics Tests Are Also Good

This test is excellent:

```python
test_compute_signature_metrics_empty_state
```

It ensures the metrics system **doesn't crash on empty data**.

Your checks are appropriate:

```
mission_status_signal exists
data_quality_signal exists
metrics_available exists
```

This ensures the system always produces a **status signal**, even without data.

That's important for executive reporting systems.

---

### The KPI test is strong too

```
test_compute_signature_metrics_with_runs_and_kpis
```

You're verifying:

```
efficiency score exists
friction score exists
status reason exists
```

This ensures the metrics engine produces **interpretable signals**, not just raw values.

---

# 5. Reporting Tests Are Well Designed

This test is exactly what you want:

```python
test_build_report_sections_produces_all_sections
```

You’re verifying that every report section exists:

```
MISSION STATUS
EXECUTIVE SUMMARY
TREND
BOTTLENECK
RISK
AGENT
PORTFOLIO
```

This ensures that **future code changes can't accidentally break report structure**.

Which is extremely valuable.

---

### The file writing test is also excellent

```
test_assemble_report_writes_file
```

You confirm:

```
file path exists
markdown file created
content contains MISSION STATUS
```

This verifies the final step of the pipeline.

---

# Small Structural Improvement

Your sys.path injection works:

```python
_root = Path(__file__).resolve().parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
```

But a cleaner approach (optional) would be:

```python
PROJECT_ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))
```

Because typically your tests sit inside:

```
tests/
```

and imports should resolve relative to the **project root**, not the test directory.

But your current solution works fine.

---

# Two High-Value Tests I Recommend Adding

These would strengthen your suite.

---

# Test 1: Recommendation Logic

You recently added **recommendation logic**, which is important.

So it deserves a test.

Example:

```python
def test_recommendation_prioritizes_risk():
    config = MissionOrchestratorV3Config()
    state = {
        "mission_status_signal": "AT RISK",
        "mission_risk_events": [{"severity": "HIGH", "risk_type": "sla_breach_risk"}],
        "task_execution_logs": [{"task_id": "T1", "actual_duration_minutes": 10}],
        "mission_runs": [],
        "telemetry_summary": {"runs_loaded": 1, "tasks_loaded": 1, "risk_events_loaded": 1},
        "mission_name": "Test Mission",
    }

    out = build_report_sections(state, config)

    assert "Recommendation:" in out["report_mission_status"]
```

This ensures your **most important decision logic** never breaks.

---

# Test 2: KPI Trend Formatting

You now show:

```
Recent values: 3.1 → 2.6 → 2.1
```

This formatting logic could break if None values appear.

So a test like this is useful:

```python
def test_trend_filters_none_values():
    config = MissionOrchestratorV3Config()

    state = {
        "mission_runs": [
            {"kpis": {"onboarding_time_days": 3.1}},
            {"kpis": {"onboarding_time_days": None}},
            {"kpis": {"onboarding_time_days": 2.1}},
        ],
        "mission_name": "Test",
    }

    out = build_report_sections(state, config)

    assert "None" not in out["report_mission_performance_trend"]
```

This protects the **clean trend formatting** you implemented.

---

# One Thing You Did That Many Engineers Forget

You are testing **behavior**, not implementation.

For example:

```
assert "MISSION STATUS" in report
```

instead of testing exact strings.

That is very good practice.

Because it allows the **report wording to evolve** without breaking tests.

---

# Final Verdict

Your test suite is **very good**.

It verifies:

```
data ingestion
signal generation
report structure
file output
```

Which are the **four critical layers** of your agent.

With the two extra tests I suggested, it becomes an **excellent test suite**.



In [ ]:
"""
Unit tests for MSO v3 orchestrator utilities: telemetry, metrics, reporting.
Use minimal or mock data; no graph. Run from project root:
  python -m pytest test_mso_v3_utilities.py -v --tb=short
"""
import sys
from pathlib import Path

_root = Path(__file__).resolve().parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import pytest

from config import MissionOrchestratorV3Config, MissionOrchestratorV3State
from agents.mso_v3.orchestrator.utilities.telemetry import load_telemetry
from agents.mso_v3.orchestrator.utilities.metrics import compute_signature_metrics
from agents.mso_v3.orchestrator.utilities.reporting import build_report_sections, assemble_report


# --- Telemetry ---


def test_load_telemetry_empty_data_dir(tmp_path):
    """With empty data dir, state gets empty lists and run_count 0."""
    config = MissionOrchestratorV3Config(data_dir=str(tmp_path), reports_dir=str(tmp_path / "out"))
    state: MissionOrchestratorV3State = {"mission_id": "M001", "errors": []}
    out = load_telemetry(state, config, project_root=tmp_path)
    assert out.get("mission_runs") == []
    assert out.get("run_count") == 0
    assert out.get("task_execution_logs") == []
    assert out.get("mission_risk_events") == []
    assert "telemetry_summary" in out
    assert out["telemetry_summary"]["runs_loaded"] == 0


def test_load_telemetry_loads_runs_and_filters_by_mission_id(tmp_path):
    """Mission runs JSON is loaded and filtered by mission_id."""
    (tmp_path / "mission_runs.json").write_text(
        '[{"run_id": "R1", "mission_id": "M001", "total_duration_minutes": 50}, '
        '{"run_id": "R2", "mission_id": "M002", "total_duration_minutes": 60}]',
        encoding="utf-8",
    )
    for f in ["task_execution_logs.json", "mission_risk_events.json", "agent_performance_stats.json"]:
        (tmp_path / f).write_text("[]", encoding="utf-8")
    config = MissionOrchestratorV3Config(data_dir=str(tmp_path), reports_dir=str(tmp_path / "out"))
    state: MissionOrchestratorV3State = {"mission_id": "M001", "errors": []}
    out = load_telemetry(state, config, project_root=tmp_path)
    assert len(out["mission_runs"]) == 1
    assert out["mission_runs"][0]["mission_id"] == "M001"
    assert out["run_count"] == 1
    assert out["telemetry_summary"]["runs_loaded"] == 1


def test_load_telemetry_sets_errors_and_run_count():
    """Errors list is initialized; run_count set from filtered runs."""
    state: MissionOrchestratorV3State = {"errors": []}
    out = load_telemetry(state, MissionOrchestratorV3Config(), project_root=Path("/nonexistent"))
    assert isinstance(out.get("errors"), list)
    assert out.get("run_count") is not None


# --- Metrics ---


def test_compute_signature_metrics_empty_state():
    """Empty state yields None metrics and a status signal."""
    config = MissionOrchestratorV3Config()
    state: MissionOrchestratorV3State = {
        "mission_runs": [],
        "task_execution_logs": [],
        "mission_risk_events": [],
        "agent_performance_stats": [],
        "mission_kpis": {},
        "run_count": 0,
    }
    out = compute_signature_metrics(state, config)
    assert "mission_status_signal" in out
    assert out["mission_status_signal"] in ("HEALTHY", "AT RISK", "OFF TARGET")
    assert "data_quality_signal" in out
    assert "metrics_available" in out


def test_compute_signature_metrics_with_runs_and_kpis():
    """With runs and KPI config, efficiency and signals are set."""
    config = MissionOrchestratorV3Config()
    state: MissionOrchestratorV3State = {
        "mission_runs": [
            {"mission_id": "M001", "total_duration_minutes": 50, "human_interventions": 2, "kpis": {"onboarding_time_days": 2.1}},
        ],
        "task_execution_logs": [
            {"task_id": "T1", "actual_duration_minutes": 10, "human_delay_minutes": 2},
        ],
        "mission_risk_events": [],
        "agent_performance_stats": [],
        "mission_kpis": {"kpi_name": "onboarding_time_days", "baseline": 5, "target": 2, "actual": 2.1, "higher_is_better": False},
        "run_count": 1,
    }
    out = compute_signature_metrics(state, config)
    assert out.get("mission_efficiency_score") is not None
    assert out.get("operational_friction_score") is not None
    assert "mission_status_reason" in out


# --- Reporting ---


def test_build_report_sections_produces_all_sections():
    """build_report_sections fills all report_* keys."""
    config = MissionOrchestratorV3Config()
    state: MissionOrchestratorV3State = {
        "mission_runs": [{"mission_id": "M001", "kpis": {"onboarding_time_days": 2.1}}],
        "mission_status_signal": "HEALTHY",
        "mission_status_reason": "All metrics within acceptable range.",
        "data_quality_signal": "HIGH",
        "data_quality_details": "1 runs loaded",
        "mission_name": "Test Mission",
        "telemetry_summary": {"runs_loaded": 1, "tasks_loaded": 0, "risk_events_loaded": 0},
    }
    out = build_report_sections(state, config)
    assert "report_mission_status" in out and "MISSION STATUS" in (out["report_mission_status"] or "")
    assert "report_executive_summary" in out and "EXECUTIVE SUMMARY" in (out["report_executive_summary"] or "")
    assert "report_mission_performance_trend" in out
    assert "report_bottleneck_analysis" in out
    assert "report_risk_analysis" in out
    assert "report_agent_performance_analysis" in out
    assert "report_portfolio_health_summary" in out


def test_assemble_report_writes_file(tmp_path):
    """assemble_report writes markdown file and sets report_file_path."""
    config = MissionOrchestratorV3Config(reports_dir=str(tmp_path / "reports"))
    state: MissionOrchestratorV3State = {
        "report_mission_status": "## MISSION STATUS\n\nMission: Test\n",
        "report_executive_summary": "## EXECUTIVE SUMMARY\n\nTest.\n",
        "report_mission_performance_trend": "## TREND\n\nNone.\n",
        "report_bottleneck_analysis": "## BOTTLENECK\n\nNone.\n",
        "report_risk_analysis": "## RISK\n\nNone.\n",
        "report_agent_performance_analysis": "## AGENT\n\nNone.\n",
        "report_portfolio_health_summary": "## PORTFOLIO\n\nNone.\n",
    }
    out = assemble_report(state, config, project_root=tmp_path)
    assert out.get("report_file_path")
    path = Path(out["report_file_path"])
    assert path.exists()
    assert "MISSION STATUS" in path.read_text(encoding="utf-8")
